# Healthcare Fraud Detection — EDA on CMS Medicare Data

**Context:** This notebook demonstrates the type of analysis underlying healthcare fraud detection AI — using public CMS Medicare data.

At Carelon (Elevance Health), I led the PM side of a similar system within a large-scale payment integrity programme. This notebook is a public illustration of the patterns and decision logic involved — not the production system.

**Key question we're exploring:** What does provider-level fraud look like in claims data, and how does ML see it differently than rule-based systems?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded.')

## Section 1: Load and Understand the Data

We're using CMS Medicare Part B Provider Utilization data — publicly available at https://data.cms.gov/provider-summary-by-type-of-service

**Key fields:**
- `nppes_provider_last_org_name` — provider name
- `nppes_provider_state` — state
- `hcpcs_code` — procedure code (CPT/HCPCS)
- `hcpcs_description` — what the procedure is
- `total_services` — number of services billed
- `average_submitted_chrg_amt` — what the provider charged
- `average_medicare_allowed_amt` — what Medicare considered reasonable
- `total_medicare_payment_amt` — what was actually paid

In [ ]:
# Download a sample of CMS Medicare data
# Full dataset: https://data.cms.gov/provider-summary-by-type-of-service
# For this demo, we'll use a local CSV or the API endpoint

# Option 1: Load from CMS API (requires internet)
# df = pd.read_csv('https://data.cms.gov/resource/fs4p-t5eq.csv?$limit=50000')

# Option 2: Load from local file (download first)
# df = pd.read_csv('cms_medicare_provider_data.csv')

# For demo purposes, create synthetic data with realistic patterns
np.random.seed(42)
n = 10000

specialties = ['Internal Medicine', 'Family Practice', 'Cardiology', 
               'Orthopedic Surgery', 'Psychiatry', 'Dermatology']
states = ['CA', 'TX', 'NY', 'FL', 'IL', 'PA', 'OH', 'GA', 'NC', 'MI']

df = pd.DataFrame({
    'provider_id': [f'NPI{i:07d}' for i in range(n)],
    'specialty': np.random.choice(specialties, n, p=[0.25, 0.25, 0.15, 0.15, 0.1, 0.1]),
    'state': np.random.choice(states, n),
    'total_services': np.random.lognormal(mean=4.5, sigma=1.2, size=n).astype(int),
    'avg_charge': np.random.lognormal(mean=5.5, sigma=0.8, size=n),
    'avg_allowed': np.random.lognormal(mean=4.8, sigma=0.6, size=n),
    'unique_beneficiaries': np.random.lognormal(mean=4.2, sigma=1.0, size=n).astype(int)
})

# Inject synthetic anomalies (what fraudulent providers look like)
anomaly_idx = np.random.choice(n, size=int(n * 0.03), replace=False)  # ~3% anomalous
df.loc[anomaly_idx, 'total_services'] = df.loc[anomaly_idx, 'total_services'] * np.random.uniform(5, 15, len(anomaly_idx))
df.loc[anomaly_idx, 'avg_charge'] = df.loc[anomaly_idx, 'avg_charge'] * np.random.uniform(3, 8, len(anomaly_idx))

df['is_anomalous'] = False
df.loc[anomaly_idx, 'is_anomalous'] = True

print(f'Dataset shape: {df.shape}')
print(f'Anomalous providers: {df.is_anomalous.sum()} ({df.is_anomalous.mean():.1%})')
df.head()

## Section 2: Provider Behavior Patterns

### PM Insight
The first question SIU investigators ask is: *"Does this provider's billing look unusual compared to their peers?"*

Legitimate fraud signals:
- Billing 10x more services than the average cardiologist in the same state
- Charging significantly above the Medicare-allowed amount for common procedures
- Seeing an implausibly high number of unique patients per day

This is what the **Provider Behavior Analytics Engine** detects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of services by specialty
for specialty in specialties:
    subset = df[df['specialty'] == specialty]['total_services']
    axes[0].hist(np.log1p(subset), alpha=0.4, label=specialty, bins=30)
axes[0].set_xlabel('log(total services)')
axes[0].set_title('Service Volume Distribution by Specialty')
axes[0].legend(fontsize=8)

# Charge-to-allowed ratio (high ratio = potential upcoding)
df['charge_ratio'] = df['avg_charge'] / df['avg_allowed']
normal = df[~df['is_anomalous']]['charge_ratio']
anomalous = df[df['is_anomalous']]['charge_ratio']

axes[1].hist(normal, bins=50, alpha=0.6, label='Normal providers', color='steelblue')
axes[1].hist(anomalous, bins=50, alpha=0.6, label='Anomalous providers', color='coral')
axes[1].set_xlabel('Charge-to-Allowed Ratio')
axes[1].set_title('Charge Ratio: Normal vs Anomalous Providers')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nMedian charge ratio — Normal: {normal.median():.2f} | Anomalous: {anomalous.median():.2f}')

## Section 3: DBSCAN Clustering — Finding Outlier Providers

### PM Insight
DBSCAN (Density-Based Spatial Clustering of Applications with Noise) is ideal here because:
1. We don't know how many "fraud clusters" exist in advance (unlike K-means)
2. Fraudulent providers often appear as isolated outliers, not members of a cluster
3. It naturally handles the fact that most providers are legitimate (high-density "normal" region)

**Points labeled -1 by DBSCAN = outliers = high priority for investigation**

In [ ]:
# Feature engineering for clustering
features = ['total_services', 'avg_charge', 'charge_ratio', 'unique_beneficiaries']
X = df[features].copy()
X = np.log1p(X)  # Log transform to handle skewed distributions

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# DBSCAN clustering
dbscan = DBSCAN(eps=0.8, min_samples=10)
df['cluster'] = dbscan.fit_predict(X_scaled)

# -1 = outlier (anomaly)
df['dbscan_outlier'] = df['cluster'] == -1

print(f'DBSCAN outliers detected: {df.dbscan_outlier.sum()} ({df.dbscan_outlier.mean():.1%})')
print(f'\nOverlap with injected anomalies:')
overlap = pd.crosstab(df['is_anomalous'], df['dbscan_outlier'], 
                       rownames=['True Anomaly'], colnames=['DBSCAN Outlier'])
print(overlap)

In [ ]:
# Visualize clusters
fig, ax = plt.subplots(figsize=(10, 7))

normal_mask = ~df['dbscan_outlier']
outlier_mask = df['dbscan_outlier']

ax.scatter(np.log1p(df[normal_mask]['total_services']), 
           np.log1p(df[normal_mask]['avg_charge']),
           alpha=0.3, s=10, label='Normal providers', color='steelblue')
ax.scatter(np.log1p(df[outlier_mask]['total_services']), 
           np.log1p(df[outlier_mask]['avg_charge']),
           alpha=0.8, s=30, label='DBSCAN outliers (investigate)', color='coral', marker='x')

ax.set_xlabel('log(Total Services)')
ax.set_ylabel('log(Average Charge)')
ax.set_title('Provider Behavior Clustering — DBSCAN Outlier Detection')
ax.legend()

plt.tight_layout()
plt.show()

## Section 4: Building an Anomaly Score

### PM Insight: Rules vs ML

Rule-based systems catch **known** fraud patterns. ML catches patterns that are statistically unusual, even if no one has written a rule for them.

**When rules are sufficient:**
- Known fraud schemes with clear thresholds (e.g., "provider billed >24 hours of service in a single day")
- Compliance requirements where the rule needs to be auditable

**When ML adds value:**
- Multi-dimensional patterns (high volume AND unusual code mix AND geographic outlier)
- Novel fraud schemes not yet in the rulebook
- Prioritization at scale — ranking 1M claims for investigators to review

In [ ]:
# Isolation Forest — unsupervised anomaly scoring
# Useful when you don't have labeled fraud examples (common in practice)

iso_forest = IsolationForest(contamination=0.03, random_state=42)
df['anomaly_score'] = iso_forest.fit_predict(X_scaled)
df['anomaly_score_raw'] = iso_forest.score_samples(X_scaled)

# Lower score = more anomalous (invert for interpretability)
df['fraud_risk_score'] = 1 - (df['anomaly_score_raw'] - df['anomaly_score_raw'].min()) / \
                              (df['anomaly_score_raw'].max() - df['anomaly_score_raw'].min())

# How well does this score separate our injected anomalies?
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(df['is_anomalous'], df['fraud_risk_score'])
print(f'AUC-ROC: {auc:.3f}')
print()

# PM metric: What % of top-100 scored cases are actual anomalies?
top_100 = df.nlargest(100, 'fraud_risk_score')
precision_at_100 = top_100['is_anomalous'].mean()
print(f'Precision@100 (top 100 flagged, % that are actual anomalies): {precision_at_100:.1%}')
print(f'-> Investigators reviewing top 100 cases would find ~{int(precision_at_100*100)} real anomalies')

# Baseline: random selection
baseline_precision = df['is_anomalous'].mean()
print(f'\nBaseline (random selection): {baseline_precision:.1%}')
print(f'Model lift: {precision_at_100/baseline_precision:.1f}x better than random')

## Section 5: PM Perspective — What These Patterns Mean for Product

### What an investigator needs (that a raw model score doesn't give them)

1. **Why was this flagged?** — The score alone isn't actionable. Which features drove it?
2. **What's the priority order?** — Investigators have limited time. Highest-value cases first.
3. **What additional data would help?** — What should they look at to confirm or rule out?

This is why the PAD application included:
- Reason codes mapped from feature contributions
- Decision Tree path visualization (not just FCNN score)
- Geographic context (same score means different things in different markets)

### The model ≠ the product

A model that scores 0.89 AUC on a holdout set can still fail as a product if:
- Investigators don't understand why cases are flagged
- The UI doesn't fit their investigation workflow
- The first few false positives erode trust before good cases can prove value

Product-level success = investigator adoption rate + net-new cases generated above baseline
Not: AUC on holdout data

In [ ]:
# Feature contribution analysis — what drove the top flagged cases?
top_flagged = df.nlargest(20, 'fraud_risk_score')[features + ['fraud_risk_score', 'specialty', 'state']]

print('Top 20 flagged providers:')
print(top_flagged[['specialty', 'state', 'total_services', 'avg_charge', 'charge_ratio', 'fraud_risk_score']]
      .to_string(index=False))

# Feature importance via correlation with anomaly score
print('\nFeature correlation with fraud risk score:')
for feat in features:
    corr = df[feat].corr(df['fraud_risk_score'])
    print(f'  {feat}: {corr:.3f}')
print()
print('-> High correlation = this feature drives the anomaly score')
print('-> This is the type of analysis that becomes "reason codes" in the investigator UI')

## Summary

This notebook demonstrated:

1. **Provider behavior patterns**: How billing volume, charge ratios, and geographic context create detectable signals
2. **DBSCAN clustering**: Why density-based clustering works for outlier detection in fraud
3. **Anomaly scoring**: Isolation Forest for unsupervised FWA detection; lift metric as the PM-relevant metric
4. **The gap between model and product**: Why explainability, workflow integration, and trust calibration matter as much as AUC

---

**Ngan Pham** — Senior PM at eBay · Previously Carelon (Elevance Health) · [LinkedIn](https://linkedin.com/in/nganpham)